In [1]:
!nvidia-smi

Wed Sep  9 12:01:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -la /content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt

-rw------- 1 root root 10524837 Aug 31 05:01 /content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt


In [4]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech
!sed -i '/norm_layer=norm_layer,/d' architecture.py

/content
Cloning into 'silent_speech'...
remote: Enumerating objects: 531, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (224/224), done.
remote: Total 531 (delta 216), reused 270 (delta 138), pack-reused 156 (from 1)
Receiving objects: 100% (531/531), 7.30 MiB | 20.83 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/content/silent_speech


In [5]:
!pip install -q flashlight-text jiwer timm torchinfo torchprofile wandb tensorboard \
  librosa soundfile noisereduce resampy praat-textgrids unidecode \
  h5py scipy joblib matplotlib tqdm requests numpy onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.0/412.0 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21

In [6]:
%cd /content/silent_speech
%env DATA_PATH=/content/data
!mkdir -p /content/data/Gaddy/h5
!cp /content/drive/MyDrive/silent_speech/emg_dataset.h5 /content/data/Gaddy/h5/ && echo "h5 OK" || echo "!! h5 missing on Drive"
!cp -r /content/drive/MyDrive/silent_speech/KenLM /content/silent_speech/ && echo "KenLM OK" || echo "!! KenLM missing on Drive"

/content/silent_speech
env: DATA_PATH=/content/data
h5 OK
KenLM OK


In [7]:
import json, os
os.chdir("/content/silent_speech")
p = "config/recognition_model.json"
cfg = json.load(open(p)); cfg["num_layers"] = 4          # this checkpoint is the 4-layer model
json.dump(cfg, open(p, "w"), indent=4)
os.environ["CK"] = "/content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt"
print("verifying:", os.environ["CK"])
!python recognition_model.py --evaluate_saved "$CK"

verifying: /content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt
2026-09-09 12:11:49.959525: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-09 12:11:49.977443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788955909.999445    3305 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788955910.005945    3305 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-09 12:11:50.027213: I tensorflow/core/platform/cpu

In [8]:
import torch, os, re, numpy as np
os.chdir("/content/silent_speech")
from architecture import EMGTransformer

CKPT = "/content/drive/MyDrive/silent_speech/output/model_20260831_044634_best.pt"
OUT  = "/content/drive/MyDrive/silent_speech/emg_ctc_fp32.onnx"

sd = torch.load(CKPT, map_location="cpu", weights_only=False)
sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
n_layers = max(int(re.match(r"blocks\.(\d+)\.", k).group(1)) for k in sd if re.match(r"blocks\.(\d+)\.", k)) + 1
num_outs = sd["w_out.weight"].shape[0]
print("layers:", n_layers, "| outputs:", num_outs)          # expect 4 | 38

model = EMGTransformer(num_features=8, num_outs=num_outs, in_chans=8, embed_dim=192,
                       n_layer=n_layers, n_head=3, mlp_ratio=4).eval()
model.load_state_dict(sd, strict=True)

class Wrap(torch.nn.Module):                                  # model ignores feat/session; feed emg 3x
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x, x, x)
wrap = Wrap(model).eval()

x = torch.randn(1, 1600, 8)
ok = False
for kw in ({"dynamo": False}, {}):                            # robust across torch versions
    try:
        torch.onnx.export(wrap, (x,), OUT, input_names=["emg"], output_names=["logits"],
            dynamic_axes={"emg": {0: "batch", 1: "time"}, "logits": {0: "batch", 1: "frames"}},
            opset_version=17, **kw)
        ok = True; break
    except TypeError as e:
        if "dynamo" in str(e): continue
        raise
assert ok, "export failed"

import onnxruntime as ort
s = ort.InferenceSession(OUT, providers=["CPUExecutionProvider"])
with torch.no_grad(): y_pt = wrap(x).numpy()
y_ox = s.run(None, {"emg": x.numpy()})[0]
print("parity max|diff|:", float(np.abs(y_pt - y_ox).max()), "(want < 1e-3)")
print("ONNX saved to Drive: %.2f MB" % (os.path.getsize(OUT) / 1e6))

layers: 4 | outputs: 38


/content/silent_speech/architecture.py:260: TracerWarning: Converting a tensor to a Python float might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  scale_factor = 1 / math.sqrt(q.size(-1))
/content/silent_speech/architecture.py:100: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  pad_length = max(length - self.max_relative_pos, 0)
/content/silent_speech/architecture.py:101: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not genera

parity max|diff|: 1.239776611328125e-05 (want < 1e-3)
ONNX saved to Drive: 10.85 MB


In [9]:
import onnxruntime as ort, numpy as np, time
s = ort.InferenceSession("/content/drive/MyDrive/silent_speech/emg_ctc_fp32.onnx", providers=["CPUExecutionProvider"])
xb = np.random.randn(1, 1600, 8).astype(np.float32)
for _ in range(3): s.run(None, {"emg": xb})
t = time.time(); [s.run(None, {"emg": xb}) for _ in range(50)]
print("mean model latency: %.1f ms (Colab CPU)" % ((time.time()-t)/50*1000))

mean model latency: 38.5 ms (Colab CPU)


In [11]:
import torch, numpy as np, onnxruntime as ort
import os; os.chdir("/content/silent_speech")
from recognition_model import test, FLAGS          # reuses the exact decoder from Cell 7
from hdf5_dataset import H5EmgDataset

class OnnxModel:                                     # drop-in stand-in for the PyTorch model
    def __init__(self, path):
        self.sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    def eval(self):  return self
    def train(self, *a, **k): return self
    def __call__(self, X, X_raw, sess):             # model uses only X_raw
        x = X_raw.detach().cpu().numpy().astype(np.float32)
        return torch.from_numpy(self.sess.run(None, {"emg": x})[0])

testset = H5EmgDataset(dev=False, test=True)
onnx_model = OnnxModel("/content/drive/MyDrive/silent_speech/emg_ctc_fp32.onnx")
wer = test(onnx_model, testset, "cpu", beam_size=1500)
print("ONNX test WER:", wer)

ONNX test WER: 0.4082624544349939
